In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 超参数定义与设备选择
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
learning_rate = 0.001
epochs = 5

# 2. 数据准备 (MNIST 数据集下载与预处理)
transform = transforms.Compose([
    transforms.ToTensor(), # 将图片转化为 Tensor，且像素值归一化到 [0, 1]
    transforms.Normalize((0.1307,), (0.3081,)) # 标准化 (均值与标准差)
])

train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

# 3. 构建多层感知机 (MLP) 全连接网络
class FCNet(nn.Module):
    def __init__(self):
        super(FCNet, self).__init__()
        # 展平后 28x28 = 784 维特征
        self.fc1 = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        # 最终输出 10 个特征/Logits，对应数字 0-9
        """
        nn.Linear(64, 10)：输入64维特征，输出10个logits
        公式：
            z = W * x + b
            W：[10, 64]权重矩阵
            b：长度为10的偏置
            输出向量z的长度=10，每个位置对应一个类别得分
        """
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        # 将输入张量 [batch_size, 1, 28, 28] 展平为 [batch_size, 784]
        x = x.view(-1, 28 * 28)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # 输出 10 个特征
        return x

model = FCNet().to(device)

# 4. 定义损失函数与优化器
criterion = nn.CrossEntropyLoss()  # 内部自动集成了 Softmax 计算
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 5. 模型训练
for epoch in range(epochs):
    """
    model.train()：把模型切换到训练模式
        Dropout启用；BN层使用batch内均值方差，并且更新滑动统计量
        只是修改self.training=True，不会开始训练，不会计算任何loss
    """
    model.train()
    """
    running_loss = 0.0：
        running_loss只是累加损失的变量，初始化为0
        用途：记录一个epoch内所有batch损失的总和，最后用来求平均损失
    """
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()        # 梯度清零
        """
        核心原理：nn.Module重写了__call__魔法方法
        class Module:
            def __call__(self, *args, **kwargs):
                # 【这里会先执行一堆钩子（hook）】
                # 然后调用你写的 forward
                result = self.forward(*args, **kwargs)
                return result
        Python里，对象加括号obj(xxx)，本质是调用这个对象的__call__方法
        """
        outputs = model(images)      # 前向传播得到 10 个特征
        loss = criterion(outputs, labels) # 计算交叉熵损失
        loss.backward()              # 反向传播
        optimizer.step()             # 更新参数
        """
        loss.item()：取出loss张量里面的Python数值，如果直接写 running_loss += loss 会构建计算图，占用大量内存！必须加.item()
            
        loss：是PyTorch张量（tensor），带有计算图、梯度信息
            
        .item()：从loss张量里面取出普通python浮点数（损失的纯数字），脱离计算图
            
        为什么放在epoch循环内：
            每一轮epoch都要清零running_loss，不然会不断累积前面所有epoch的损失，数值越来越大
        """
        running_loss += loss.item()
    
    # len(train_loader) = 一个epoch里面一共有多少个batch
    # running_loss / len(train_loader)：求一个epoch里的平均损失
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# 6. 模型测试 (验证识别准确率)
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        # logits（原始得分）
        outputs = model(images)
        
        # 通过 argmax 取 10 个输出特征中最大值的索引，即为预测数字
        """
        在深度学习训练和预测中，模型通常是批次（Batch）输入数据的，因此输出张量（outputs）是一个2维张量，其形状为[batch_size, 10]：
            维度0（dim=0）：代表Batch（样本）方向。沿着这个方向找最大值，会得到一个Batch里的所有图片在某个类别上的最大得分（通常没有实际预测意义）
            维度1（dim=1）：代表Classes（类别）方向，也就是对应数字0到9的10个得分特征
        因此，传入1就是告诉PyTorch：对每张图片（按行遍历），在它的10个类别得分中找到最大值
        """
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"测试集准确率: {100 * correct / total:.2f}%")

print(type(outputs))

Epoch [1/5], Loss: 0.4793
Epoch [2/5], Loss: 0.3494
Epoch [3/5], Loss: 0.3163
Epoch [4/5], Loss: 0.2913
Epoch [5/5], Loss: 0.2748
测试集准确率: 88.06%
<class 'torch.Tensor'>


In [9]:
"""
直观对比示例：
    假设Batch大小为2（有两张图片），模型输出outputs如下：
"""
# outputs 的形状是 [2, 10]  [batch_size, num_classes]
outputs = torch.tensor([
    [0.1, 0.2, 0.0, 0.85, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],  # 第 1 张图片，索引 3 数值最大
    [0.0, 0.9, 0.1, 0.00, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]   # 第 2 张图片，索引 1 数值最大
])
"""
values：每一行最大数值/最高得分是多少
predicted：最大数值所在索引位置（index），即预测的数字是几（0～9）
torch.max()与torch.argmax()的区别：
    相同点：
        都是Pytorch中寻找张量（Tensor）中最大值的函数，两者的核心区别在于返回值内容不同
    核心区别：
        torch.max：既返回最大值本身（values），也返回最大值所在的索引（indices）
        torch.argmax：只返回最大值所在的索引（indices）
"""
values, predicted = torch.max(outputs, 1)

print(values)     # 输出: tensor([0.85, 0.90]) -> 对应的最大得分
print(predicted)  # 输出: tensor([3, 1])      -> 沿着 dim=1 找到的最大值索引（即预测结果）

tensor([0.8500, 0.9000])
tensor([3, 1])


In [11]:
print(len(train_dataset))
print(type(train_dataset))

print(len(train_loader))
print(type(train_loader))

60000
<class 'torchvision.datasets.mnist.FashionMNIST'>
938
<class 'torch.utils.data.dataloader.DataLoader'>


In [7]:
60000 / 64

937.5